# FiveCrop features — the five missing PAIP slides

The five slides that were never in the feature set:

`training_data_19_nonMSIH` · `training_data_30_MSIH` · `training_data_41_nonMSIH` · `training_data_42_MSIH` · `training_data_46_nonMSIH`

Their patches were copied into `paip_data\patch_data`, so it now has 78 slide folders instead of 73.

**Read this before running anything.** The other fivecrop notebooks all filter patches through
`final_merged_without_white.csv`, and that file has no rows for these five slides — it was built from the
73-slide run. Point any of those notebooks at these slides and you get "No valid non-white patches found"
and an empty folder. So we build the filter first:

1. **Metadata** — RGB mean/std for every patch → `patches_metadata\<slide>.csv`
2. **White filter** — the RBF SVM from `data_cleaning/PAIP_SVM`, keep `label == 0` → appended to `final_merged_without_white.csv`
3. **Features** — `FiveCrop(256)` on each 512×512 patch → `(5, D)` per patch

Steps 1 and 2 are CPU only. Step 3 needs the GPU. Everything skips work that is already done, so you can
re-run any cell.

> PRISM is **not** here — it needs its own environment. Use `Feature_Extraction_Missing_PAIP_PRISM.ipynb`
> after step 2.

In [1]:
import os
import re
import glob
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

PAIP_ROOT      = r"D:\Aamir Gulzar\KSA_project2\paip_data"
PATCH_DATA_DIR = os.path.join(PAIP_ROOT, "patch_data")
METADATA_DIR   = os.path.join(PAIP_ROOT, "patches_metadata")
NON_WHITE_CSV  = os.path.join(PAIP_ROOT, "final_merged_without_white.csv")
FEATURES_ROOT  = os.path.join(PAIP_ROOT, "Features")

REPO_ROOT = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
SVM_MODEL  = os.path.join(REPO_ROOT, "data_cleaning", "PAIP_SVM", "svm_model_rbf.pkl")
SVM_SCALER = os.path.join(REPO_ROOT, "data_cleaning", "PAIP_SVM", "scaler_rbf.pkl")
CONCH_V1_CKPT = os.path.join(REPO_ROOT, "feature_extraction", "checkpoints",
                             "CONCH", "pytorch_model.bin")

SLIDES = [
    "training_data_19_nonMSIH",
    "training_data_30_MSIH",
    "training_data_41_nonMSIH",
    "training_data_42_MSIH",
    "training_data_46_nonMSIH",
]

CROP_SIZE = 256   # FiveCrop(256) on a 512x512 patch, same as the other notebooks

for s in SLIDES:
    n = len(glob.glob(os.path.join(PATCH_DATA_DIR, s, "*.png")))
    print(f"{s:<26} {n:>6,} patches")

training_data_19_nonMSIH      915 patches
training_data_30_MSIH       1,525 patches
training_data_41_nonMSIH      170 patches
training_data_42_MSIH       1,118 patches
training_data_46_nonMSIH      609 patches


---
## Step 1 — patch metadata

Same 11 columns as the CSVs the other 73 slides already have.

In [2]:
# Patch names come in two flavours, because the five slides were patched by an
# older run:
#   training_data_01_MSIH_x10240_y44776_patch00004.png   (the 73)
#   training_data_30_MSIH_x10240_y0_450.png              (the five)
NAME_RE = re.compile(r"_x(\d+)_y(\d+)_(?:patch)?(\d+)$")

COLUMNS = ["patch_location", "patch_name", "patch_number", "patch_x", "patch_y",
           "avg_R", "avg_G", "avg_B", "std_R", "std_G", "std_B"]

# Averaging 262,144 pixels in float32 loses enough precision to move the std by
# ~0.2, so the precision has to match whatever the existing CSVs were made with.
# The next cell works out which one that is and sets this.
STATS_DTYPE = np.float64


def patch_stats(path, dtype=None):
    """Per-channel mean and std over the raw 0-255 pixels."""
    img = np.asarray(Image.open(path).convert("RGB"), dtype=dtype or STATS_DTYPE)
    return img.mean(axis=(0, 1)), img.std(axis=(0, 1))


def build_metadata(slide):
    patches = sorted(glob.glob(os.path.join(PATCH_DATA_DIR, slide, "*.png")))
    if not patches:
        print(f"  no patches for {slide}")
        return

    rows = []
    for i, path in enumerate(tqdm(patches, desc=slide, unit="patch", leave=False)):
        name = os.path.basename(path)
        m = NAME_RE.search(os.path.splitext(name)[0])
        x, y, number = (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else (None, None, i)

        avg, std = patch_stats(path)
        rows.append([path, name, number, x, y,
                     round(float(avg[0]), 2), round(float(avg[1]), 2), round(float(avg[2]), 2),
                     round(float(std[0]), 2), round(float(std[1]), 2), round(float(std[2]), 2)])

    df = pd.DataFrame(rows, columns=COLUMNS)
    os.makedirs(METADATA_DIR, exist_ok=True)
    out = os.path.join(METADATA_DIR, slide + ".csv")
    df.to_csv(out, index=False)
    print(f"  {slide}: {len(df):,} patches -> {out}")

In [3]:
# Sanity check: recompute the stats for a few patches of a slide that is already
# done and compare with its stored CSV. If they don't line up, we're computing
# numbers on a different basis than the SVM was trained on.
#
# float32 and float64 disagree slightly over 262,144 pixels, so try both and keep
# whichever the existing CSVs were written with. The stored values are rounded to
# 2 decimals, so the winner should land under 0.01.

done_csvs = [f for f in sorted(os.listdir(METADATA_DIR))
             if f.endswith(".csv") and f[:-4] not in SLIDES]

df = pd.read_csv(os.path.join(METADATA_DIR, done_csvs[0]))
sample = df.sample(50, random_state=0)

worst = {}
for dtype in (np.float32, np.float64):
    w = 0
    for _, row in sample.iterrows():
        avg, std = patch_stats(row["patch_location"], dtype)
        stored = row[["avg_R", "avg_G", "avg_B", "std_R", "std_G", "std_B"]].to_numpy(float)
        w = max(w, np.abs(np.concatenate([avg, std]) - stored).max())
    worst[dtype] = w
    print(f"{np.dtype(dtype).name}: largest difference {w:.4f}")

STATS_DTYPE = min(worst, key=worst.get)
print(f"\nchecked 50 patches from {done_csvs[0]}")
print(f"using {np.dtype(STATS_DTYPE).name} - matches the existing CSVs")

assert worst[STATS_DTYPE] < 0.01, (
    "neither precision matches the stored CSVs, so something else differs "
    "(different images, or the stats were computed some other way) - stop here"
)

float32: largest difference 0.1986
float64: largest difference 0.0050

checked 50 patches from training_data_01_MSIH.csv
using float64 - matches the existing CSVs


In [4]:
for slide in SLIDES:
    if os.path.exists(os.path.join(METADATA_DIR, slide + ".csv")):
        print(f"  {slide}: already done")
    else:
        build_metadata(slide)

  training_data_19_nonMSIH: 915 patches -> D:\Aamir Gulzar\KSA_project2\paip_data\patches_metadata\training_data_19_nonMSIH.csv


  training_data_30_MSIH: 1,525 patches -> D:\Aamir Gulzar\KSA_project2\paip_data\patches_metadata\training_data_30_MSIH.csv


  training_data_41_nonMSIH: 170 patches -> D:\Aamir Gulzar\KSA_project2\paip_data\patches_metadata\training_data_41_nonMSIH.csv


  training_data_42_MSIH: 1,118 patches -> D:\Aamir Gulzar\KSA_project2\paip_data\patches_metadata\training_data_42_MSIH.csv


  training_data_46_nonMSIH: 609 patches -> D:\Aamir Gulzar\KSA_project2\paip_data\patches_metadata\training_data_46_nonMSIH.csv


---
## Step 2 — white patch filter

Same as `data_cleaning/PAIP_SVM/white_patch_removal_PAIP.ipynb`: scale the three columns, predict,
keep `label == 0`. Label 1 means white/background.

The CSV is backed up first, and any old rows for these slides are dropped before appending, so running
this twice won't duplicate anything.

In [5]:
import joblib

svm    = joblib.load(SVM_MODEL)
scaler = joblib.load(SVM_SCALER)
FEATURE_COLS = ["avg_B", "std_G", "std_B"]

kept_frames = []
for slide in SLIDES:
    df = pd.read_csv(os.path.join(METADATA_DIR, slide + ".csv"))
    df["label"] = svm.predict(scaler.transform(df[FEATURE_COLS].values))

    kept = df[df["label"] == 0]
    print(f"{slide:<26} keep {len(kept):>6,} / drop {len(df) - len(kept):>6,} "
          f"({(len(df) - len(kept)) / len(df):.1%} white)")
    kept_frames.append(kept)

new_rows = pd.concat(kept_frames, ignore_index=True)
print(f"\ntotal new rows: {len(new_rows):,}")

training_data_19_nonMSIH   keep    898 / drop     17 (1.9% white)
training_data_30_MSIH      keep  1,466 / drop     59 (3.9% white)
training_data_41_nonMSIH   keep    170 / drop      0 (0.0% white)
training_data_42_MSIH      keep  1,116 / drop      2 (0.2% white)
training_data_46_nonMSIH   keep    569 / drop     40 (6.6% white)

total new rows: 4,219


In [6]:
# back up, drop any old rows for these slides, append the new ones
backup = f"{NON_WHITE_CSV}.bak-{datetime.now():%Y%m%d-%H%M%S}"
shutil.copy2(NON_WHITE_CSV, backup)
print("backup:", backup)

existing = pd.read_csv(NON_WHITE_CSV)
prefixes = tuple(s + "_" for s in SLIDES)
existing = existing[~existing["patch_name"].astype(str).str.startswith(prefixes)]

merged = pd.concat([existing, new_rows], ignore_index=True)
merged.to_csv(NON_WHITE_CSV, index=False)
print(f"{NON_WHITE_CSV}: {len(merged):,} rows")

backup: D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv.bak-20260809-183948
D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv: 275,541 rows


---
## Step 3 — the extractors

Each class is the one from its own notebook: five 256×256 crops of the patch, each through the model's own
transform, stacked to `(5, 3, 224, 224)`, one forward pass, `(5, D)` back on the CPU.

Run only the models you need — each one pulls down a few GB of weights.

In [7]:
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from timm.layers import SwiGLUPacked
from torchvision import transforms
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not set - UNI2, Virchow2 and H-optimus are gated repos")
login(token=hf_token)
print("logged in to hugging face")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
logged in to hugging face


In [8]:
class UNI2FiveCrop:
    """UNI2-h -> (5, 1536)"""

    def __init__(self, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        kwargs = {
            "img_size": 224, "patch_size": 14, "depth": 24, "num_heads": 24,
            "init_values": 1e-5, "embed_dim": 1536, "mlp_ratio": 2.66667 * 2,
            "num_classes": 0, "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked, "act_layer": torch.nn.SiLU,
            "reg_tokens": 8, "dynamic_img_size": True,
        }
        self.model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **kwargs)
        self.model = self.model.to(self.device).eval()

        config = resolve_data_config(self.model.pretrained_cfg, model=self.model)
        self.eval_transform = create_transform(**config)
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])

    def extract_fivecrop_features(self, path):
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)
        with torch.no_grad():
            return self.model(crops).cpu()

In [9]:
class ConchV1FiveCrop:
    """CONCH v1 -> (5, 512). Loads from the local checkpoint."""

    def __init__(self, device="cuda"):
        from conch.open_clip_custom import create_model_from_pretrained

        model, preprocess = create_model_from_pretrained("conch_ViT-B-16", CONCH_V1_CKPT)
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.conch = model.to(self.device).eval()
        self.eval_transform = preprocess
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])

    def extract_fivecrop_features(self, path):
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)
        with torch.no_grad():
            return self.conch.encode_image(crops, proj_contrast=False, normalize=False).cpu()


class ConchV15FiveCrop:
    """CONCH v1.5 - the tile encoder that ships inside TITAN."""

    def __init__(self, device="cuda"):
        from transformers import AutoModel

        titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
        self.conch, self.eval_transform = titan.return_conch()

        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.conch = self.conch.to(self.device).eval()
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])

    def extract_fivecrop_features(self, path):
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)
        with torch.no_grad():
            return self.conch(crops).cpu()

In [10]:
class Virchow2FiveCrop:
    """Virchow2 -> (5, 2560): class token joined with the mean patch token."""

    def __init__(self, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model = timm.create_model(
            "hf-hub:paige-ai/Virchow2", pretrained=True,
            mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU,
        ).to(self.device).eval()

        cfg = resolve_data_config(self.model.pretrained_cfg, model=self.model)
        self.eval_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=cfg["mean"], std=cfg["std"]),
        ])
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])

    @torch.inference_mode()
    def extract_fivecrop_features(self, path):
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            out = self.model(crops)          # (5, 261, 1280)

        cls_token = out[:, 0]                # (5, 1280)
        patches   = out[:, 5:]               # (5, 256, 1280) - tokens 1-4 are registers
        return torch.cat([cls_token, patches.mean(dim=1)], dim=-1).cpu().float()


class HOptimusFiveCrop:
    """H-optimus-1 -> (5, 1536)"""

    def __init__(self, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model = timm.create_model(
            "hf-hub:bioptimus/H-optimus-1", pretrained=True,
            init_values=1e-5, dynamic_img_size=False,
        ).to(self.device).eval()

        self.eval_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.707223, 0.578729, 0.703617),
                                 std=(0.211883, 0.230117, 0.177517)),
        ])
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])

    def extract_fivecrop_features(self, path):
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            with torch.inference_mode():
                return self.model(crops).cpu()

### The loop

Same as the other notebooks, just limited to our five slides instead of walking all 78 folders.

In [11]:
# folder names are the ones already holding the 73 slides
MODELS = {
    "uni2h":       (UNI2FiveCrop,     "Uni_2h_fivecrop_Updated"),
    "conch_v1":    (ConchV1FiveCrop,  "Conch_v1_fivecrop_Updated"),
    "conch_v1_5":  (ConchV15FiveCrop, "Conch_v1_5_fivecrop_Updated"),
    "virchow2":    (Virchow2FiveCrop, "Virchow2_fivecrop_Updated"),
    "h_optimus_1": (HOptimusFiveCrop, "H_Optimus_1_fivecrop_Updated"),
}


def run_model(name, slides=SLIDES, device="cuda"):
    cls, folder = MODELS[name]
    output_root = os.path.join(FEATURES_ROOT, folder)
    print(f"{name} -> {output_root}")

    valid_names = set(pd.read_csv(NON_WHITE_CSV)["patch_name"].astype(str))
    extractor = cls(device=device)

    for slide in slides:
        out_dir = os.path.join(output_root, slide)
        os.makedirs(out_dir, exist_ok=True)

        patches = glob.glob(os.path.join(PATCH_DATA_DIR, slide, "*.png"))
        patches = [p for p in patches if os.path.basename(p) in valid_names]
        if not patches:
            print(f"  {slide}: nothing to do - did step 2 run?")
            continue

        done = {f[:-3] for f in os.listdir(out_dir) if f.endswith(".pt")}
        todo = [p for p in patches if os.path.splitext(os.path.basename(p))[0] not in done]
        if not todo:
            print(f"  {slide}: all {len(patches):,} patches already extracted")
            continue

        shape = None
        for path in tqdm(todo, desc=f"  {slide}", unit="patch", leave=False):
            stem = os.path.splitext(os.path.basename(path))[0]
            features = extractor.extract_fivecrop_features(path)
            torch.save(features, os.path.join(out_dir, stem + ".pt"))
            shape = tuple(features.shape)

        print(f"  {slide}: {len(todo):,} extracted, shape {shape}")

    del extractor
    torch.cuda.empty_cache()

In [12]:
run_model("uni2h")

uni2h -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Uni_2h_fivecrop_Updated


  training_data_19_nonMSIH:   0%|          | 0/898 [00:00<?, ?patch/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(


  training_data_19_nonMSIH: 898 extracted, shape (5, 1536)


  training_data_30_MSIH: 1,466 extracted, shape (5, 1536)


  training_data_41_nonMSIH: 170 extracted, shape (5, 1536)


  training_data_42_MSIH: 1,116 extracted, shape (5, 1536)


  training_data_46_nonMSIH: 569 extracted, shape (5, 1536)


In [13]:
run_model("conch_v1")

conch_v1 -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_fivecrop_Updated


c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


  training_data_19_nonMSIH: 898 extracted, shape (5, 512)


  training_data_30_MSIH: 1,466 extracted, shape (5, 512)


  training_data_41_nonMSIH: 170 extracted, shape (5, 512)


  training_data_42_MSIH: 1,116 extracted, shape (5, 512)


  training_data_46_nonMSIH: 569 extracted, shape (5, 512)


In [14]:
run_model("conch_v1_5")

conch_v1_5 -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_5_fivecrop_Updated


  training_data_19_nonMSIH: 898 extracted, shape (5, 768)


  training_data_30_MSIH: 1,466 extracted, shape (5, 768)


  training_data_41_nonMSIH: 170 extracted, shape (5, 768)


  training_data_42_MSIH: 1,116 extracted, shape (5, 768)


  training_data_46_nonMSIH: 569 extracted, shape (5, 768)


In [15]:
run_model("virchow2")

virchow2 -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Virchow2_fivecrop_Updated


  training_data_19_nonMSIH: 898 extracted, shape (5, 2560)


  training_data_30_MSIH: 1,466 extracted, shape (5, 2560)


  training_data_41_nonMSIH: 170 extracted, shape (5, 2560)


  training_data_42_MSIH: 1,116 extracted, shape (5, 2560)


  training_data_46_nonMSIH: 569 extracted, shape (5, 2560)


In [16]:
run_model("h_optimus_1")

h_optimus_1 -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\H_Optimus_1_fivecrop_Updated


config.json:   0%|          | 0.00/452 [00:00<?, ?B/s]

c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\datainsight\.cache\huggingface\hub\models--bioptimus--H-optimus-1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

  training_data_19_nonMSIH: 898 extracted, shape (5, 1536)


  training_data_30_MSIH: 1,466 extracted, shape (5, 1536)


  training_data_41_nonMSIH: 170 extracted, shape (5, 1536)


  training_data_42_MSIH: 1,116 extracted, shape (5, 1536)


  training_data_46_nonMSIH: 569 extracted, shape (5, 1536)


---
## Check the results

Each folder should now hold 78 slides, and the five new ones should have the same tensor shape as the 73
and one `.pt` per non-white patch.

In [17]:
non_white = pd.read_csv(NON_WHITE_CSV)["patch_name"].astype(str)

for name, (_, folder) in MODELS.items():
    root = os.path.join(FEATURES_ROOT, folder)
    n_slides = len([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])

    # what shape do the already-finished slides have?
    old_slide = next(d for d in sorted(os.listdir(root)) if d not in SLIDES)
    old_file  = next(f for f in os.listdir(os.path.join(root, old_slide)) if f.endswith(".pt"))
    ref_shape = tuple(torch.load(os.path.join(root, old_slide, old_file),
                                 map_location="cpu", weights_only=True).shape)

    print(f"\n{name}  ({n_slides} slides, reference shape {ref_shape})")

    for slide in SLIDES:
        d = os.path.join(root, slide)
        pts = [f for f in os.listdir(d) if f.endswith(".pt")] if os.path.isdir(d) else []
        if not pts:
            print(f"  {slide:<26} EMPTY")
            continue

        t = torch.load(os.path.join(d, pts[0]), map_location="cpu", weights_only=True)
        expected = int(non_white.str.startswith(slide + "_").sum())
        ok = tuple(t.shape) == ref_shape and len(pts) == expected and torch.isfinite(t).all()
        print(f"  {slide:<26} {len(pts):>6,} / {expected:<6,} {tuple(t.shape)} "
              f"{'ok' if ok else 'CHECK'}")


uni2h  (78 slides, reference shape (5, 1536))
  training_data_19_nonMSIH      898 / 898    (5, 1536) ok
  training_data_30_MSIH       1,466 / 1,466  (5, 1536) ok
  training_data_41_nonMSIH      170 / 170    (5, 1536) ok
  training_data_42_MSIH       1,116 / 1,116  (5, 1536) ok
  training_data_46_nonMSIH      569 / 569    (5, 1536) ok

conch_v1  (78 slides, reference shape (5, 512))
  training_data_19_nonMSIH      898 / 898    (5, 512) ok
  training_data_30_MSIH       1,466 / 1,466  (5, 512) ok
  training_data_41_nonMSIH      170 / 170    (5, 512) ok
  training_data_42_MSIH       1,116 / 1,116  (5, 512) ok
  training_data_46_nonMSIH      569 / 569    (5, 512) ok

conch_v1_5  (78 slides, reference shape (5, 768))
  training_data_19_nonMSIH      898 / 898    (5, 768) ok
  training_data_30_MSIH       1,466 / 1,466  (5, 768) ok
  training_data_41_nonMSIH      170 / 170    (5, 768) ok
  training_data_42_MSIH       1,116 / 1,116  (5, 768) ok
  training_data_46_nonMSIH      569 / 569    (5, 7